# Train the colony detector (RF-DETR Medium) on Kaggle

**Settings panel (right side) before running:**
- Accelerator: **GPU T4 x2** (or P100)
- Internet: **On** (needs a phone-verified Kaggle account)
- Persistence: *Files only* (optional)

Then **Save Version → Save & Run All (Commit)**. It runs in the background (you can close the tab),
takes about 8–9 hours, and everything ends up under **Output** (`/kaggle/working`):
`rfdetr_tiles.pth`, `detector.json`, `test_results.csv` and the training log.

In [ ]:
# 1. GPU, code, packages
import torch, os
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))
OUT = '/kaggle/working'
!rm -rf /kaggle/temp/colony_detector && git clone -q https://github.com/akashbabu9392/colony_detector.git /kaggle/temp/colony_detector
%cd /kaggle/temp/colony_detector
!pip -q install -r requirements.txt "rfdetr[train]>=1.7" pyyaml

In [ ]:
# 2. Dataset (~3.2 GB) from the GitHub release
!mkdir -p /kaggle/temp/data && cd /kaggle/temp/data && \
  wget -q -nc https://github.com/akashbabu9392/colony_detector/releases/download/dataset-v1/data.set.zip && \
  wget -q -nc https://github.com/akashbabu9392/colony_detector/releases/download/dataset-v1/data.set.2.zip && \
  unzip -q -n data.set.zip -d raw && unzip -q -n data.set.2.zip -d raw && rm -f *.zip && ls raw

In [ ]:
# 3. Full-resolution tiles + whole-plate views (test split held out)  (~15-25 min)
!python tools/build_training_set.py --dataset /kaggle/temp/data/raw --out /kaggle/temp/tiles --scale 1.0 \
    --empty-keep 0.15 --coco --gold "" | tail -1

In [ ]:
# 4. Train RF-DETR Medium: 10 epochs x ~45 min on a T4 fits Kaggle's 12 h limit.
!python tools/train_rfdetr.py --dataset /kaggle/temp/tiles/coco --size medium --resolution 640 \
    --epochs 10 --batch 4 --grad-accum 4 --lr 1e-4 --output /kaggle/temp/rfdetr --out models/rfdetr_tiles.pth \
    2>&1 | grep -E --line-buffered "Epoch|Best|mAP|Error|Traceback|->" | tee {OUT}/train_log.txt
!cp models/rfdetr_tiles.pth {OUT}/

In [ ]:
# 5. Fit the confidence cut-off on VALIDATION plates, measure on TEST plates
os.environ['CD_RFDETR_WEIGHTS'] = 'models/rfdetr_tiles.pth'
!python tools/tune_detector.py --dataset /kaggle/temp/data/raw --engine rfdetr --split valid --conf-only \
    --cache /kaggle/temp/valid_dets.pkl | tail -4 | tee {OUT}/tune_log.txt
!CD_ENGINES=rfdetr python tools/evaluate.py --dataset /kaggle/temp/data/raw --split test \
    --out {OUT}/test_results.csv | tail -25 | tee {OUT}/test_summary.txt
!cp models/detector.json {OUT}/ && ls -la {OUT}